# 💻 Práctica 6: Detección de objetos

En esta práctica vamos a trabajar con los conceptos centrales de detección de objetos que vimos en la teoría.

El objetivo es implementar y evaluar las métricas que usamos para medir el desempeño de un detector:

- Calcular el IoU entre cajas y usarla para clasificar detecciones como TP o FP.
- Construir la curva Precision-Recall a partir de las predicciones y su score de confianza.
- Calcular el Average Precision (AP).

Además, vamos a correr un detector YOLOv8 preentrenado y evaluaremos su desempeño en un subconjunto de COCO.

📌 Antes de comenzar, generá una copia de este archivo en tu Drive y trabajá en esa copia.

---

## 🧭 Parte 1: Práctica guiada

### Setup inicial

Instalamos las dependencias necesarias para la práctica: `Ultralytics` para usar YOLO y `pycocotools` para evaluar modelos detección con métricas COCO.

In [ ]:
!pip install ultralytics pycocotools --quiet

Importamos las bibliotecas que vamos a usar a lo largo de la práctica:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random
import json
import os
from pathlib import Path
from collections import defaultdict

import torch
from torchvision.datasets import CocoDetection
import torchvision.transforms as T

from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

### Intersection over Union (IoU)

El IoU mide la superposición entre bounding boxes. Es el criterio que utilizamos para decidir si una predicción es TP o FP.

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

Las cajas se representarán en formato **xyxy**: `[x1, y1, x2, y2]` donde:

- `(x1, y1)` es la esquina superior izquierda y
- `(x2, y2)` la inferior derecha.

In [ ]:
def compute_iou(box_a, box_b):
    """
    Calcula el IoU entre dos cajas en formato xyxy.
    box_a, box_b: [x1, y1, x2, y2]
    """
    # Intersección
    ix1 = max(box_a[0], box_b[0])
    iy1 = max(box_a[1], box_b[1])
    ix2 = min(box_a[2], box_b[2])
    iy2 = min(box_a[3], box_b[3])

    inter_w = max(0, ix2 - ix1)
    inter_h = max(0, iy2 - iy1)
    inter_area = inter_w * inter_h

    # Unión
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union_area = area_a + area_b - inter_area

    if union_area == 0:
        return 0.0

    return inter_area / union_area

Verifiquemos la función con ejemplos con IoU conocido:

In [ ]:
# Coincidencia perfecta: IoU = 1.0
box = [10, 10, 50, 50]
print(f"Coincidencia perfecta: IoU = {compute_iou(box, box):.4f}")

# Sin solapamiento: IoU = 0
box_a = [0, 0, 10, 10]
box_b = [20, 20, 30, 30]
print(f"Sin solapamiento: IoU = {compute_iou(box_a, box_b):.4f}")

# Solapamiento parcial
box_gt = [100, 100, 200, 200]  # 100x100
box_pred = [150, 150, 250, 250]  # desplazada 50px
iou = compute_iou(box_gt, box_pred)
print(f"Solapamiento parcial: IoU = {iou:.4f}")

Ahora visualicemos en un gráfico el efecto del desplazamiento en el valor de IoU:

In [ ]:
gt_box = [50, 50, 150, 150]  # caja GT fija
offsets = list(range(0, 130, 5))  # desplazamientos a evaluar
ious = []

# Calculamos el IoU
for offset in offsets:
    pred = [50 + offset, 50 + offset, 150 + offset, 150 + offset]
    ious.append(compute_iou(gt_box, pred))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Izquierda: curva IoU vs offset
axes[0].plot(offsets, ious, color="steelblue", linewidth=2)
axes[0].axhline(0.5, color="tomato", linestyle="--", label="umbral 0.50 (PASCAL VOC)")
axes[0].axhline(0.75, color="orange", linestyle="--", label="umbral 0.75 (COCO strict)")
axes[0].set_xlabel("Desplazamiento de la predicción (px)")
axes[0].set_ylabel("IoU")
axes[0].set_title("IoU en función del desplazamiento")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Derecha: cajas para un offset concreto
offset_example = 40
pred_box = [50 + offset_example, 50 + offset_example, 150 + offset_example, 150 + offset_example]
iou_ex = compute_iou(gt_box, pred_box)

axes[1].set_xlim(0, 220)
axes[1].set_ylim(0, 220)
axes[1].set_aspect("equal")

rect_gt = patches.Rectangle((gt_box[0], gt_box[1]),
    gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
    linewidth=2, edgecolor="green", facecolor="green", alpha=0.25, label="Objeto (GT)")
rect_pred = patches.Rectangle((pred_box[0], pred_box[1]),
    pred_box[2]-pred_box[0], pred_box[3]-pred_box[1],
    linewidth=2, edgecolor="tomato", facecolor="tomato", alpha=0.25, label="Predicción")

# Intersección
ix1 = max(gt_box[0], pred_box[0])
iy1 = max(gt_box[1], pred_box[1])
ix2 = min(gt_box[2], pred_box[2])
iy2 = min(gt_box[3], pred_box[3])

rect_inter = patches.Rectangle((ix1, iy1), ix2-ix1, iy2-iy1,
    linewidth=0, facecolor="purple", alpha=0.4, label="Intersección")

axes[1].add_patch(rect_gt)
axes[1].add_patch(rect_pred)
axes[1].add_patch(rect_inter)
axes[1].set_title(f"Offset={offset_example}px: IoU={iou_ex:.3f}")
axes[1].legend()
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

❓ Preguntas:

1. ¿Qué valor de IoU tiene una predicción cuya caja es el doble del tamaño del GT y está centrada sobre él?
2. ¿Un umbral IoU de 0.5 es un criterio estricto o permisivo?
3. ¿Qué problema tiene el IoU si los objetos son muy alargados o están rotados?

### Clasificación de detecciones: TP, FP

Dado un umbral de IoU `t`, clasificamos cada predicción como TP o FP:

- Las predicciones se recorren **de mayor a menor score**.
- La primera predicción que supere `IoU ≥ t` con un GT libre es **TP**. Ese GT queda ocupado.
- El resto de predicciones que apunten al mismo GT son **FP**.

Además, recordemos que cualquier GT sin ninguna predicción asignada se considera **FN**.

In [ ]:
def match_predictions(predictions, ground_truths, iou_threshold=0.5):
    """
    Clasifica predicciones como TP o FP usando el protocolo estándar.

    predictions: lista de dicts con keys "box" (xyxy) y "score"
    ground_truths: lista de cajas GT en formato xyxy
    iou_threshold: umbral para considerar una detección como correcta

    Retorna:
        tp: array binario (1=TP, 0=FP) para cada predicción
        scores: scores correspondientes
        n_gt: cantidad de ground truths
    """

    # Ordenamos predicciones de mayor a menor score
    preds_sorted = sorted(predictions, key=lambda x: x["score"], reverse=True)

    # GTs ya asignados
    matched_gt = set()
    tp = []
    scores = []

    for pred in preds_sorted:
        best_iou = 0.0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(ground_truths):
            if gt_idx in matched_gt:
                continue
            iou = compute_iou(pred["box"], gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_threshold and best_gt_idx not in matched_gt:
            tp.append(1)
            matched_gt.add(best_gt_idx)
        else:
            tp.append(0)

        scores.append(pred["score"])

    return np.array(tp), np.array(scores), len(ground_truths)

Probemos clasificar predicciones con un ejemplo de imagen con 2 GTs y 3 predicciones:

In [ ]:
ground_truths_ex = [
    [10, 10, 60, 60], # GT 0
    [80, 80, 140, 140], # GT 1
    [200, 200, 220, 280], # GT 2
]

predictions_ex = [
    {"box": [12, 12, 58, 58], "score": 0.95},  # buena sobre GT 0
    {"box": [82, 82, 138, 138], "score": 0.88},  # buena sobre GT 1
    {"box": [15, 15, 55, 55], "score": 0.72},  # duplicado de GT 0
]

tp, scores, n_gt = match_predictions(predictions_ex, ground_truths_ex, iou_threshold=0.5)

print("Scores (orden desc.):", scores)
print("TP/FP:", tp, "(1=TP, 0=FP)")
print(f"GTs totales: {n_gt} | TPs: {tp.sum()} | FPs: {(1-tp).sum()} | FNs: {n_gt - tp.sum()}")

❓ Preguntas:

1. ¿Por qué se recorren las predicciones en orden de score descendente?
2. Si un modelo detecta el mismo objeto 5 veces con scores distintos, ¿cuántos TP y cuántos FP genera?
3. ¿A qué corresponde el FN que se obtuvo?

### Curva Precision-Recall

La curva PR se construye acumulando TP y FP en orden de score descendente.

En cada paso calculamos Precision y Recall acumulado:

$$P_n = \frac{\text{acc TP}_n}{\text{total Pred}_n} \qquad R_n = \frac{\text{acc TP}_n}{\text{total GT}}$$

In [ ]:
def compute_pr_curve(tp_array, n_gt):
    """
    Calcula la curva PR a partir del array de TP/FP (ya ordenado por score desc.)
    y el número total de ground truths.
    """
    acc_tp = np.cumsum(tp_array)
    n_preds = np.arange(1, len(tp_array) + 1)

    precision = acc_tp / n_preds
    recall = acc_tp / n_gt

    # Agregar punto inicial (0, 1) para cerrar la curva
    precision = np.concatenate([[1.0], precision])
    recall = np.concatenate([[0.0], recall])

    return precision, recall

Generamos un conjunto sintético de detecciones para visualizar la curva completa:

In [ ]:
# Simulamos un detector con 30 predicciones: los primeros son buenos, luego peores
scores_demo = np.sort(np.random.uniform(0.1, 1.0, 30))[::-1]

# TPs concentrados al inicio (buena precisión), luego más FPs
n_gt_demo = 20
tp_demo = np.array([1,1,0,1,1,0,0,1,1,0,1,0,0,1,1,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0])

precision_demo, recall_demo = compute_pr_curve(tp_demo, n_gt_demo)

plt.figure(figsize=(8, 5))
plt.step(recall_demo, precision_demo, where="post", color="steelblue", linewidth=2, label="Curva PR")
plt.fill_between(recall_demo, precision_demo, alpha=0.15, color="steelblue", step="post")
plt.scatter(recall_demo[1:], precision_demo[1:], s=30, color="steelblue", zorder=5)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Curva Precision-Recall (ejemplo sintético)")
plt.xlim([0, 1.05])
plt.ylim([0, 1.05])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Recall máximo alcanzado: {recall_demo.max():.3f}")
print(f"Precision en recall=0: {precision_demo[0]:.3f}")

❓ Preguntas:

1. ¿Qué forma debería tener la curva PR de un detector perfecto? ¿Y la de uno aleatorio?
2. ¿Qué ocurre con la curva PR si bajamos el umbral de confianza? ¿Y si lo subimos?


### Average Precision (AP)

El AP es el área bajo la curva PR. Para calcularlo, vamos a usar el método *all-points* (VOC 2010+).

Con este método, para cada punto de recall tomamos la precisión máxima a la derecha y calculamos el área como suma de escalones.

$$\text{AP} = \sum_n (R_n - R_{n-1}) \cdot P_{\text{interp}}(R_n)$$

donde $P_{\text{interp}}(R_n) = \max_{R' \geq R_n} P(R')$.

In [ ]:
def compute_ap(precision, recall):
    """
    Calcula el AP usando la interpolación all-points (VOC 2010+).
    Aplica la interpolación por la derecha para eliminar dientes de sierra.
    """
    # Interpolación: en cada punto tomar la precisión máxima hacia la derecha
    precision_interp = np.maximum.accumulate(precision[::-1])[::-1]

    # Calcular cambios de recall
    recall_diff = np.diff(recall)

    # AP = suma de (delta_recall * precision_interpolada)
    ap = np.sum(recall_diff * precision_interp[1:])

    return ap

Calculemos el AP y grafiquemos la curva real y la interpolada:

In [ ]:
ap_demo = compute_ap(precision_demo, recall_demo)

print(f"AP (all-points): {ap_demo:.4f}")

precision_interp = np.maximum.accumulate(precision_demo[::-1])[::-1]

plt.figure(figsize=(8, 5))
plt.step(recall_demo, precision_demo, where="post", color="lightblue",
    linewidth=1.5, label="Curva PR original", linestyle="--")
plt.step(recall_demo, precision_interp, where="post", color="steelblue",
    linewidth=2, label="Curva interpolada")
plt.fill_between(recall_demo, precision_interp, alpha=0.2, color="steelblue", step="post",
    label=f"AP = {ap_demo:.4f}")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("AP - Interpolación all-points")
plt.xlim([0, 1.05])
plt.ylim([0, 1.05])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### mAP sobre múltiples clases

El **mAP** promedia el AP sobre todas las clases y nos sirve para resumir el desempeño del detector.

Vamos a implementar una función que recibe las predicciones y GTs de múltiples imágenes organizados por clase.

In [ ]:
def compute_map(all_predictions, all_ground_truths, iou_threshold=0.5):
    """
    Calcula mAP sobre múltiples clases.

    all_predictions: dict {class_id: [{"box": xyxy, "score": float, "image_id": int}, ...]}
    all_ground_truths: dict {class_id: {image_id: [box_xyxy, ...]}}
    iou_threshold: umbral IoU para TP

    Retorna: mAP (float), ap_per_class (dict)
    """
    ap_per_class = {}

    # Iterar sobre cada clase presente en las predicciones
    for class_id, preds in all_predictions.items():

        # Número total de objetos reales (GT) en esta clase
        gts_by_image = all_ground_truths.get(class_id, {})
        n_gt = sum(len(boxes) for boxes in gts_by_image.values())

        # Si no hay GTs para esta clase, se ignora
        if n_gt == 0:
            continue

        # Ordenar predicciones por score (de mayor a menor confianza)
        preds_sorted = sorted(preds, key=lambda x: x["score"], reverse=True)

        matched = defaultdict(set)

        # Lista donde guardamos TP (1) o FP (0) en orden de score
        tp_list = []

        # Evaluamos cada predicción en orden de confianza
        for pred in preds_sorted:

            img_id = pred["image_id"]
            gt_boxes = gts_by_image.get(img_id, [])

            best_iou = 0.0
            best_gt_idx = -1

            # Comparamos predicción contra todos los GT de la misma imagen
            for gt_idx, gt_box in enumerate(gt_boxes):

                # Si el GT ya fue asignado a otra predicción, se ignora
                if gt_idx in matched[img_id]:
                    continue

                # Calcular IoU entre predicción y GT
                iou = compute_iou(pred["box"], gt_box)

                # Guardar el mejor match (mayor IoU)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = gt_idx

            # Si supera el umbral IoU: TP
            if best_iou >= iou_threshold and best_gt_idx >= 0:
                tp_list.append(1)  # True Positive
                matched[img_id].add(best_gt_idx)

            else:
                tp_list.append(0)  # FP

        tp_arr = np.array(tp_list)

        # Curva PR
        precision, recall = compute_pr_curve(tp_arr, n_gt)

        # AP
        ap = compute_ap(precision, recall)
        ap_per_class[class_id] = ap

    if not ap_per_class:
        return 0.0, {}

    # mAP
    map_value = np.mean(list(ap_per_class.values()))

    return map_value, ap_per_class

Probemos con un ejemplo de 3 clases y 3 imágenes:

In [ ]:
all_preds = {
    "cat": [
        {"box": [10,10,60,60], "score": 0.95, "image_id": 0},  # IoU≈1.0: TP
        {"box": [90,90,145,148], "score": 0.82, "image_id": 1},  # IoU≈0.71: TP@50, FP@75
        {"box": [5,5,55,55], "score": 0.60, "image_id": 0},  # duplicado
    ],
    "dog": [
        {"box": [20,20,70,70], "score": 0.90, "image_id": 0},  # IoU≈1.0: TP
        {"box": [200,200,260,260],"score": 0.75, "image_id": 2},  # sin GT
    ],
    "car": [
        {"box": [45,45,85,85], "score": 0.88, "image_id": 1},  # IoU≈0.41: TP@25, FP@50
        {"box": [100,100,180,180],"score": 0.55, "image_id": 2},  # IoU≈1.0: TP
    ],
}

all_gts = {
    "cat": {0: [[10, 10, 60, 60]], 1: [[80, 80, 140, 140]]},
    "dog": {0: [[20, 20, 70, 70]]},
    "car": {1: [[30, 30, 80, 80]], 2: [[100, 100, 180, 180]]},
}

map_val, ap_by_class = compute_map(all_preds, all_gts, iou_threshold=0.75)

print("AP por clase:")
for cls, ap in ap_by_class.items():
    print(f"  {cls:10s}: {ap:.4f}")

print(f"\nmAP@50: {map_val:.4f}")

❓ Preguntas:

1. ¿Qué problema tiene el mAP cuando las clases tienen cantidades muy distintas en el dataset?
2. Si una clase tiene AP = 0 porque nunca fue detectada, ¿cómo afecta esto al mAP?
3. ¿Qué pasa si modificamos el umbral de IoU?

### YOLOv8 en COCO val2017

Ahora vamos a probar correr un detector real: **YOLOv8n**

Usaremos un modelo preentrenado en COCO y lo evaluaremos sobre un subconjunto de 100 imágenes de `val2017`.

Primero, descargamos las imágenes y las anotaciones:

In [ ]:
# Anotaciones COCO val2017
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip
print("Anotaciones descargadas.")

In [ ]:
# Cargamos anotaciones y seleccionamos 100 imágenes al azar
coco_gt = COCO("annotations/instances_val2017.json")

all_img_ids = coco_gt.getImgIds()
selected_ids = np.random.choice(all_img_ids, size=100, replace=False).tolist()

print(f"Imágenes seleccionadas: {len(selected_ids)}")
print(f"Ejemplo de image_ids: {selected_ids[:10]}")

In [ ]:
# Descargamos solo las imágenes seleccionadas
import urllib.request

os.makedirs("val2017_subset", exist_ok=True)
img_infos = coco_gt.loadImgs(selected_ids)

base_url = "http://images.cocodataset.org/val2017/"
for info in img_infos:
    fname = info["file_name"]
    dst = f"val2017_subset/{fname}"
    if not os.path.exists(dst):
        urllib.request.urlretrieve(base_url + fname, dst)

print(f"Imágenes descargadas: {len(os.listdir("val2017_subset"))}")

Cargamos YOLOv8n preentrenado y lo probamos:

In [ ]:
model = YOLO("yolov8n.pt")
print(model.info(verbose=False))

In [ ]:
img_infos = coco_gt.loadImgs(selected_ids)
img_paths_ordered = [Path("val2017_subset") / info["file_name"] for info in img_infos]

# Hacemos inferencia sobre las 100 imágenes
results = model.predict(img_paths_ordered, conf=0.25, verbose=False)

coco_results = []
for r, img_id in zip(results, selected_ids):
    if r.boxes is None:
        continue
    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
        yolo_cls = int(box.cls[0])
        coco_results.append({
            "image_id": img_id,
            "category_id": YOLO_TO_COCO_ID.get(yolo_cls, yolo_cls + 1),
            "bbox": [x1, y1, x2-x1, y2-y1],
            "score": float(box.conf[0])
        })

In [ ]:
# Visualizamos 6 ejemplos
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, (ax, r) in enumerate(zip(axes, results[:6])):
    img_bgr = r.orig_img
    img_rgb = img_bgr[:, :, ::-1]
    ax.imshow(img_rgb)
    ax.axis("off")

    if r.boxes is not None:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            score = float(box.conf[0])
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                linewidth=1.5, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1-3, f"{cls_name} {score:.2f}",
                    color="lime", fontsize=7, fontweight="bold")
    ax.set_title(f"img {i+1}", fontsize=9)

plt.suptitle("YOLOv8n - detecciones en val2017 (subconjunto)", fontsize=12)
plt.tight_layout()
plt.show()

Evaluamos los resultados y extraemos las métricas:

In [ ]:
# Evaluar con COCOeval
coco_dt = coco_gt.loadRes(coco_results)

coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.params.imgIds = selected_ids
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

In [ ]:
# Extraemos las métricas para visualizar
metrics = {
    "AP@50:95": coco_eval.stats[0],
    "AP@50": coco_eval.stats[1],
    "AP@75": coco_eval.stats[2],
    "AP-S": coco_eval.stats[3],
    "AP-M": coco_eval.stats[4],
    "AP-L": coco_eval.stats[5],
}

print("\nMétricas COCO (YOLOv8n, 100 imágenes val2017)")
for name, val in metrics.items():
    print(f"  {name:<12}: {val:.4f}")

# Gráfico de barras
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metrics.keys(), metrics.values(),
              color=["steelblue","steelblue","steelblue","tomato","orange","teal"])

ax.set_ylim(0, 1)
ax.set_ylabel("AP")
ax.set_title("Métricas COCO — YOLOv8n (subset 100 imgs)")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
    f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

❓ Preguntas:

1. Comparando AP@50 y AP@75, ¿qué dice esa diferencia sobre la calidad de localización del modelo?
2. ¿Por qué AP-S es más bajo que AP-L?

---

## 🧠 Parte 2: Para resolver

En esta segunda parte vamos a explorar un poco más sobre IoU y el umbral de confianza.

### Ejercicio 1: Sensibilidad del AP al umbral de IoU

El umbral de IoU determina qué tan estrictos somos con la localización.

PASCAL VOC usa 0.50 (permisivo) y COCO promedia entre 0.50 y 0.95 (más completo).

**Consigna:**

1. Usando las predicciones y GTs del ejemplo sintético de la parte guiada, calculá el AP para cada clase usando los umbrales `[0.25, 0.50, 0.75, 0.90]`.
2. Graficá el AP en función del umbral para cada clase en un mismo plot (una línea por clase).
3. Calculá también el **mAP@50:95** sobre esos umbrales (promedio entre 0.50 y 0.95 con paso 0.05, como hace COCO).
4. Analizá qué clase tiene la caída más pronunciada al aumentar el umbral.

In [ ]:
# Umbrales a evaluar
# thresholds = [0.25, 0.50, 0.75, 0.90]

# Calcular AP por clase para cada umbral
# ap_results = {cls: [] for cls in all_preds.keys()}

# for t in thresholds:
#     _, ap_by_class = compute_map(all_preds, all_gts, iou_threshold=t)
#     for cls, ap in ap_by_class.items():
#         ap_results[cls].append(ap)

# Graficar AP vs umbral por clase

# Calcular mAP@50:95
# thresholds_coco = np.arange(0.50, 1.00, 0.05)
# map_50_95 = ...

### Ejercicio 2: Efecto del umbral de confianza (score)

El umbral de confianza determina qué predicciones se reportan.

Un umbral alto nos da pocas detecciones pero más precisas, mientras que un umbral bajo da más detecciones pero menos precisas.

**Consigna:**

1. Tomá los resultados de YOLOv8 sobre el subconjunto de 100 imágenes.
2. Para cada umbral en `[0.10, 0.25, 0.50, 0.75]`:
   - Filtrá las detecciones con `score ≥ umbral`.
   - Calculá la cantidad promedio de detecciones por imagen.
3. Graficá el número promedio de detecciones vs umbral de confianza.
4. Elegí una imagen y visualizá sus detecciones con los 4 umbrales.

In [ ]:
# conf_thresholds = [0.10, 0.25, 0.50, 0.75]
# avg_dets = []

# for conf in conf_thresholds:
#     results_conf = model.predict(img_paths, conf=conf, verbose=False)
#     n_dets = [len(r.boxes) if r.boxes is not None else 0 for r in results_conf]
#     avg_dets.append(np.mean(n_dets))

# Graficar avg_dets vs conf_thresholds

# Visualizar una imagen a los 4 umbrales

---

## 📚 Recursos

- Python: https://docs.python.org/3/tutorial/
- NumPy: https://numpy.org/learn/
- Matplotlib: https://matplotlib.org/stable/index.html
- Google Colab: https://colab.research.google.com/

- Ultralytics YOLOv8: https://docs.ultralytics.com/
- pycocotools: https://github.com/cocodataset/cocoapi
- COCO dataset: https://cocodataset.org/

- COCO evaluation: https://cocodataset.org/#detection-eval
- Padilla et al. (2021) - A Comparative Analysis of Object Detection Metrics: https://www.mdpi.com/2079-9292/10/3/279